# Crypto.Net · 04 — Solana

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./04-solana.id.ipynb)

Phantom-compatible accounts, key formats, live devnet reads, program-derived addresses and a simulated transaction with Compute Budget, System and Memo instructions.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Wallet;
using Crypto.Net.Solana;

var wallet  = HdWallet.Generate();
var account = wallet.GetSolanaAccount(0, SolanaChain.Devnet);   // m/44'/501'/0'/0' (Phantom, Solflare)
Console.WriteLine($"{account.Address} ({account.DerivationPath})");
Console.WriteLine($"Phantom secret key format: {account.ExportBase58SecretKey()[..12]}… (64 bytes)");
Console.WriteLine($"solana-cli keypair JSON:   {account.ExportKeypairJson()[..40]}…");

## Live devnet

In [ ]:
var rpc = new SolanaRpcClient(SolanaChain.Devnet);
var (blockhash, lastValid) = await rpc.GetLatestBlockhashAsync();
Console.WriteLine($"slot {await rpc.GetBlockNumberAsync():N0}, blockhash {blockhash} (valid until height {lastValid:N0})");
Console.WriteLine($"balance {await rpc.GetBalanceAsync(account.Address)} SOL");
Console.WriteLine($"rent-exempt minimum for 165 bytes: {Amount.FromLamports(await rpc.GetMinimumBalanceForRentExemptionAsync(165))} SOL");

## Program-derived addresses

PDAs are off the ed25519 curve, so no private key exists for them. The associated token account (ATA) is the PDA every wallet uses for a given mint (here: devnet USDC).

In [ ]:
const string DevnetUsdc = "4zMMC9srt5Ri5X14GAgXhaHii3GnPAEERYPJgZJDncDU";
string ata = SolanaAddress.GetAssociatedTokenAddress(account.Address.Value, DevnetUsdc);
var (vault, bump) = SolanaAddress.FindProgramAddress([SolanaAddress.Utf8Seed("vault"), SolanaAddress.Decode(account.Address.Value)], SolanaAddress.MemoProgram);

new[]
{
    new { What = "wallet", Address = account.Address.Value, OnCurve = SolanaAddress.IsOnCurve(account.Address.Value) },
    new { What = "USDC ATA", Address = ata, OnCurve = SolanaAddress.IsOnCurve(ata) },
    new { What = $"PDA (bump {bump})", Address = vault, OnCurve = SolanaAddress.IsOnCurve(vault) },
}

## Build, sign and simulate

A fresh account has no SOL, so the simulation reports an error — that is the node validating our signed transaction. Fund it with `RequestAirdropAsync` (devnet) to see it succeed.

In [ ]:
var tx = new SolanaTransaction(account.Address.Value, blockhash)
    .Add(ComputeBudgetProgram.SetComputeUnitPrice(1_000))
    .Add(SystemProgram.Transfer(account.Address.Value, "11111111111111111111111111111112", 5_000))
    .Add(MemoProgram.Memo("Hello from Crypto.Net", account.Address.Value));
account.Sign(tx);

byte[] wire = tx.Serialize();
Console.WriteLine($"signature (tx id) {tx.Signature}");
Console.WriteLine($"{wire.Length} bytes · accounts: {string.Join(", ", tx.CompileAccounts().Select(a => a.PublicKey[..6] + (a.IsSigner ? "(s)" : "") + (a.IsWritable ? "(w)" : "")))}");

var (error, logs) = await rpc.SimulateTransactionAsync(wire);
Console.WriteLine(error is null ? "simulation succeeded" : $"simulation error: {error}");
foreach (var line in logs) Console.WriteLine($"  {line}");

## Sending for real

```csharp
await rpc.RequestAirdropAsync(account.Address.Value, 1_000_000_000); // devnet, rate-limited
TxHash sig = await rpc.TransferAsync(account, "<recipient>", Amount.FromSol(0.1m), priorityFeeMicroLamports: 1_000);
TxHash spl = await rpc.TransferTokenAsync(account, DevnetUsdc, "<recipient>", Amount.Parse("1", 6));
```

In [ ]:
account.Dispose(); rpc.Dispose(); wallet.Dispose();
"disposed"